# Lead-Lag Analysis

I compared monthly vessel returns with VIX levels, VIX returns, and WTI returns at leads of 0–12 months. Lag 3, for example, compares WTI returns with vessel returns three months later. This is an exploratory correlation scan, not a forecasting test.

In [ ]:
from src.lead_lag import (
    build_lead_lag_table,
    get_best_lag,
    plot_combined_lead_lag,
)

In [ ]:
lead_lag_results = build_lead_lag_table(max_lag=12)

lead_lag_results

In [ ]:
lead_lag_results

## Selecting Candidate Lags

For each indicator, the helper first looks for lags with a raw p-value below 0.05 and selects the largest absolute correlation among them; if none meet that threshold, it selects the largest absolute correlation overall. This selection rule also uses the observed p-values, so its apparent significance should not be treated as an independent test. The p-values in the table are **unadjusted**. Because the selection comes after comparing 13 lags per indicator, a p-value below 0.05 at the selected lag is not enough to establish significance across the search. For WTI, a simple Bonferroni adjustment across its 13 tested lags gives approximately 0.412 (0.031681 × 13). This conservative check also does not address possible serial dependence in monthly data.

In [ ]:
selected_features = [
    "vix_level",
    "vix_return",
    "wti_return",
]

best_lag_rows = []

for feature in selected_features:
    best_result, is_significant = get_best_lag(
        result_df=lead_lag_results,
        feature_name=feature,
    )

    best_lag_rows.append(
        {
            "feature": feature,
            "best_lag": int(best_result["lag_months"]),
            "correlation": best_result["correlation"],
            "p_value": best_result["p_value"],
            "observations": int(best_result["observations"]),
            "significant_at_5pct": is_significant,
        }
    )

In [ ]:
import pandas as pd

best_lag_summary = pd.DataFrame(best_lag_rows)

best_lag_summary

## Lead-Lag Correlation Curves

The chart shows VIX levels and WTI returns across the tested lags. VIX returns are omitted for readability. Markers identify the strongest *observed* correlation for each series, not a confirmed leading indicator.

In [ ]:
plot_combined_lead_lag(
    result_df=lead_lag_results,
    feature_names=[
        "vix_level",
        "wti_return",
    ],
)

## Key Findings

WTI returns have their largest observed correlation with vessel returns at a three-month lead (r = 0.156, unadjusted p = 0.032; n = 190). The relationship is weak and does not remain significant under a Bonferroni adjustment across the 13 WTI lags. WTI at two months is close behind (r = 0.141), so three months should not be treated as a precise or stable lead time.

VIX level peaks in absolute correlation at a two-month lead (r = -0.128, unadjusted p = 0.078). VIX returns have no comparably notable peak. These are candidates for further exploration, not validated forecasting signals.

## Business Interpretation

WTI may partly reflect broader economic and commodity-market conditions relevant to shipping. The correlations here are too small, however, to support a practical early-warning claim on their own. They also cannot establish causation.

I carry the selected WTI and VIX lags into a regression to examine their joint association with vessel returns. Because the lags were selected on this same sample, those regression p-values also need to be read as exploratory.

## Next Step

The next notebook fits OLS models using WTI returns at lag 3 and VIX levels at lag 2. Out-of-sample testing and time-series-robust inference would be needed before treating either as a reliable monitoring signal.